# Gradient Boosting

Boosting **secuencial** donde cada arbol se entrena para predecir los **residuos** del modelo acumulado hasta ese momento; el `learning_rate` pondera el aporte de cada arbol. A diferencia de AdaBoost (que reweightea observaciones), aca cada modelo ajusta directamente el error que queda.

Regresion sobre `Hitters`, target = `log(Salary)`, comparado contra la regresion lineal base.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error

## Datos

`Hitters` sin nulos, solo variables numericas. Target = `log(Salary)`.

In [ ]:
data_complete = pd.read_csv("../datasets/Hitters.csv").dropna()

data_columns=['AtBat','Hits','HmRun','Runs','RBI','Walks', 'Years', 'CAtBat', 'CHits', 'CHmRun', 'CRuns', 'CRBI', 'CWalks',
                'PutOuts', 'Assists', 'Errors', 'Salary']
data = data_complete.loc[:, data_columns]

X = data.drop("Salary", axis=1)
y = np.log(data.Salary)

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=127)

In [ ]:
scaler = StandardScaler()
X_train_scl = scaler.fit_transform(X_train)
X_test_scl = scaler.transform(X_test)

## Referencia: regresion lineal sola

In [ ]:
base_regressor = LinearRegression()
fit_base = base_regressor.fit(X_train_scl, y_train)
predict_base = fit_base.predict(X_test_scl)
performance_base = mean_squared_error(y_test, predict_base)
print(performance_base)

## Gradient Boosting

`GradientBoostingRegressor` con 200 arboles de profundidad 4 y `learning_rate=0.5`. Al usar arboles (estimadores debiles) el boosting si aprovecha bien la secuencia.

In [ ]:
gb_reg = GradientBoostingRegressor(loss='squared_error', n_estimators=200, learning_rate=0.5, max_depth=4, random_state=127, subsample=1)

In [ ]:
gb_fit = gb_reg.fit(X_train_scl, y_train)

In [ ]:
prediction = gb_reg.predict(X_test_scl)
performance = mean_squared_error(y_test, prediction)
performance

MSE base ~0.414 vs Gradient Boosting ~0.262. Con arboles como estimador base, el boosting reduce el error de forma clara respecto de la regresion lineal (y respecto de AdaBoost con base lineal).